# Caso 06: AulaViva — Cursos online: ¿Qué alumno se queda?
## Día 1: Bienvenida a AulaViva y Primer Contacto con los Datos (EDA Forense)

**Objetivo de hoy:**
1. **Entender el modelo de negocio de suscripción:**
   - AulaViva es una plataforma edtech por suscripción (plan mensual de `14,99 €` y anual de `119,00 €`, catálogo de 600 cursos en 6 categorías). En suscripción, **captar no es retener**: el negocio no gana dinero cuando el alumno se da de alta, sino cuando aprende, renueva y deja valor a largo plazo (**LTV**).
   - **La tensión de la semana:** Diego Losada (Director de Growth) afirma que *"el cupón es el número uno"* (más altas y menor CAC) y quiere volcar el presupuesto del trimestre en escalarlo. Elena Sanchís (CEO) y Nuria Aguilar (Lead de Analítica) necesitan saber qué ocurre después del alta: si los alumnos del cupón aprenden, renuevan y generan valor real.
2. **El modelo en 4 actos (el alumno en el centro):**
   - **Acto 1 — Captación:** `usuarios.csv`, `canales.csv`, `campanas.csv` (lo que mira Growth).
   - **Acto 2 — Aprendizaje:** `matriculas.csv`, `cursos.csv` (lo que mira Producto Educativo).
   - **Acto 3 — Permanencia:** `suscripciones.csv` (si el alumno sigue activo o causó baja).
   - **Acto 4 — Valor:** `pagos.csv` (cuánto deja cada alumno entre altas y renovaciones).
3. **Las 5 preguntas del primer vistazo y las 3 minas de Toni Márquez:**
   - **1. FORMA (`.shape`):** Escala de las tablas de hechos frente a las tablas de dimensiones y detección de duplicados.
   - **2. TIPOS (`.dtypes`):** Lectura como radiografía (importes, porcentajes y fechas leídos como texto; IDs leídos como `float64` por presencia de vacíos).
   - **3. NULOS (`.isnull().sum()`):** Separar **nulos con significado** (`campana_id` vacío = alta orgánica/referido; `fecha_fin` vacía = suscripción activa) de **nulos reales** (`metodo` de pago vacío = desconocido).
   - **4. GRAFÍAS (`.nunique()` / `.value_counts()`):** Cuántas variantes ortográficas esconden los 5 canales, los planes, los estados y las categorías.
   - **5. ¿CUADRA?:** Comprobar que las magnitudes coinciden con el negocio antes de tocar una sola cifra.
   - **Regla de oro de hoy:** *Hoy NO se limpia ni se calcula ninguna tasa de negocio; hoy se audita el terreno.*


In [1]:
import pandas as pd
import numpy as np
import os
import duckdb

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RAW_DIR = os.path.join('data', 'raw')
print('Directorio de datos crudos:', RAW_DIR)


Directorio de datos crudos: data\raw


In [2]:
def auditoria_tabla(df, nombre, id_col=None, max_unique_show=25):
    print(f"{'='*20} AUDITORÍA: {nombre} {'='*20}")
    print(f"1. FORMA: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    dups_exactos = df.duplicated().sum()
    print(f"   -> Duplicados exactos de fila: {dups_exactos:,}")
    if id_col and id_col in df.columns:
        dups_id = df[id_col].duplicated().sum()
        print(f"   -> Duplicados por clave primaria ('{id_col}'): {dups_id:,} (IDs únicos: {df[id_col].nunique():,})")
        
    print(f"\n2. TIPOS DETECTADOS (Radiografía):")
    for col, dtype in df.dtypes.items():
        sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else 'N/A'
        print(f"   - {col:22s} {str(dtype):10s} (Ejemplo: {repr(sample_val)})")
        
    print(f"\n3. NULOS / VACÍOS (.isnull().sum()):")
    nulls = df.isnull().sum()
    if nulls.sum() == 0:
        print("   -> Sin valores nulos en ninguna columna.")
    else:
        for col, cnt in nulls[nulls > 0].items():
            pct = (cnt / len(df)) * 100
            print(f"   - {col:22s}: {cnt:,} nulos ({pct:.2f}%)")
            
    print(f"\n4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):")
    text_cols = df.select_dtypes(include=['object', 'str']).columns
    for col in text_cols:
        nunique = df[col].nunique()
        print(f"   - {col:22s}: {nunique} valores distintos")
        if nunique <= max_unique_show:
            val_counts = df[col].value_counts(dropna=False).to_dict()
            print(f"     Frecuencias: {val_counts}")
    print("\n")


### 1. Auditoría: `usuarios.csv` (El Centro del Modelo — Acto 1: Captación)
En el centro del modelo está el alumno: cada fila representa una persona dada de alta, por qué canal llegó y, si procedía de una campaña de pago, su `campana_id`.


In [3]:
df_usuarios = pd.read_csv(os.path.join(RAW_DIR, 'usuarios.csv'), encoding='utf-8')
auditoria_tabla(df_usuarios, 'usuarios.csv', id_col='usuario_id')
df_usuarios.head(5)


==================== AUDITORÍA: usuarios.csv ====================
1. FORMA: 40,000 filas x 6 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('usuario_id'): 0 (IDs únicos: 40,000)

2. TIPOS DETECTADOS (Radiografía):
   - usuario_id             int64      (Ejemplo: np.int64(1))
   - canal                  str        (Ejemplo: 'Anuncios')
   - campana_id             float64    (Ejemplo: np.float64(1024.0))
   - fecha_registro         str        (Ejemplo: '2024-04-09')
   - ciudad                 str        (Ejemplo: 'sevilla')
   - plan_inicial           str        (Ejemplo: 'mensual')

3. NULOS / VACÍOS (.isnull().sum()):
   - campana_id            : 14,255 nulos (35.64%)
   - ciudad                : 800 nulos (2.00%)

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - canal                 : 19 valores distintos
     Frecuencias: {'cupon': 6828, 'anuncios': 4438, 'Cupón': 4244, 'seo': 3373, 'descuento': 2360, 'Anuncios': 2320, 'referido': 2116, 'organi

,usuario_id,canal,campana_id,fecha_registro,ciudad,plan_inicial
0,1,Anuncios,1024.0,2024-04-09,NaN,mensual
1,2,seo,NaN,10/05/2023,sevilla,año
2,3,referido,NaN,2 Dec 2022,Sevilla,MES
3,4,cupon,1005.0,2024-05-28,Barcelona,mensual
4,5,Anuncios,NaN,01/06/2024,SVQ,Mensual


**Hallazgos forenses en `usuarios.csv`:**
* **1. Forma y Cuadre:** `40,000` filas x `6` columnas (`40,000` alumnos únicos, `0` duplicados). Cuadra exactamente con la escala de ~40.000 altas.
* **2. Radiografía de Tipos:**
  * `campana_id` aparece como `float64` en lugar de entero: **bandera roja de vacíos** en un identificador.
  * `fecha_registro` aparece como texto (`object`/`str`): mezcla de formatos (`YYYY-MM-DD`, `DD/MM/YYYY` y fechas con mes en inglés como `"12 Jun 2023"`).
* **3. Nulos con significado vs. Nulos reales:**
  * `campana_id` tiene **14,255 vacíos (35.64%)**: tal como avisó Toni, **NO es un dato perdido**, sino altas orgánicas (SEO), de referido o sin campaña asociada. Hacer un `INNER JOIN` con `campanas.csv` o borrar estos nulos destruiría canales enteros.
  * `ciudad` tiene **800 vacíos (2.00%)**: nulo informativo menor.
* **4. Grafías descontroladas:**
  * `canal` está escrito de **19 formas distintas** (ej. `cupon`, `Cupón`, `CUPON`, `descuento`, `Anuncios`, `publicidad`, `seo`, `organico`, `referido`, `amigo`, `empresas`, `corporativo`, `b2b`...) cuando el negocio solo tiene **5 canales reales**.
  * `plan_inicial` tiene **8 grafías** para solo 2 planes (`mensual`, `Mensual`, `MES`, `mes`, `anual`, `Anual`, `año`, `12m`...).
  * `ciudad` tiene **20 grafías** (`Madrid`, `MAD`, `madrid`, `Barcelona`, `BCN`, `Sevilla`, `SVQ`...).


### 2. Auditoría: `canales.csv` (Dimensión Principal de Captación)
Catálogo maestro de los 5 canales oficiales de AulaViva y su naturaleza (`pagado`, `organico`, `referido`, `b2b`).


In [4]:
df_canales = pd.read_csv(os.path.join(RAW_DIR, 'canales.csv'), encoding='utf-8')
auditoria_tabla(df_canales, 'canales.csv', id_col='canal_id')
df_canales


==================== AUDITORÍA: canales.csv ====================
1. FORMA: 5 filas x 3 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('canal_id'): 0 (IDs únicos: 5)

2. TIPOS DETECTADOS (Radiografía):
   - canal_id               int64      (Ejemplo: np.int64(1))
   - nombre                 str        (Ejemplo: 'cupon')
   - tipo                   str        (Ejemplo: 'pagado')

3. NULOS / VACÍOS (.isnull().sum()):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - nombre                : 5 valores distintos
     Frecuencias: {'cupon': 1, 'anuncios': 1, 'seo': 1, 'referido': 1, 'empresas': 1}
   - tipo                  : 4 valores distintos
     Frecuencias: {'pagado': 2, 'organico': 1, 'Referido': 1, 'B2B': 1}




,canal_id,nombre,tipo
0,1,cupon,pagado
1,2,anuncios,pagado
2,3,seo,organico
3,4,referido,Referido
4,5,empresas,B2B


**Hallazgos forenses en `canales.csv`:**
* **1. Forma:** Tabla de dimensión diminuta (`5 filas x 3 columnas`), sin duplicados ni nulos.
* **2. Los 5 canales canónicos:** `cupon`, `anuncios`, `seo`, `referido` y `empresas`. Todos los valores sucios de `usuarios.canal` y `campanas.canal` deberán mapearse a estos 5 nombres exactos en el Día 2.
* **3. Grafías en `tipo`:** Mezcla de mayúsculas/minúsculas (`pagado`, `organico`, `Referido`, `B2B`).


### 3. Auditoría: `campanas.csv` (Dimensión de Coste — Acto 1: Captación)
Registra las campañas de marketing con su coste de inversión. Contesta «cuánto costó captar» y solo existe para los canales de pago (`cupon`, `anuncios`, `empresas`).


In [5]:
df_campanas = pd.read_csv(os.path.join(RAW_DIR, 'campanas.csv'), encoding='utf-8')
auditoria_tabla(df_campanas, 'campanas.csv', id_col='campana_id')
df_campanas.head(5)


==================== AUDITORÍA: campanas.csv ====================
1. FORMA: 45 filas x 7 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('campana_id'): 0 (IDs únicos: 45)

2. TIPOS DETECTADOS (Radiografía):
   - campana_id             int64      (Ejemplo: np.int64(1001))
   - canal                  str        (Ejemplo: 'descuento')
   - nombre                 str        (Ejemplo: 'Cupon Bienvenida')
   - fecha_inicio           str        (Ejemplo: '18/05/2023')
   - fecha_fin              str        (Ejemplo: '12/11/2023')
   - coste                  str        (Ejemplo: '5.397,33 €')
   - tipo                   str        (Ejemplo: 'Descuento')

3. NULOS / VACÍOS (.isnull().sum()):
   - fecha_fin             : 3 nulos (6.67%)

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - canal                 : 14 valores distintos
     Frecuencias: {'cupon': 6, 'empresas': 6, 'Cupón': 5, 'publicidad': 5, 'descuento': 4, 'CUPON': 3, 'ads': 3, 'referido': 3, 'an

,campana_id,canal,nombre,fecha_inicio,fecha_fin,coste,tipo
0,1001,descuento,Cupon Bienvenida,18/05/2023,12/11/2023,"5.397,33 €",Descuento
1,1002,Cupón,Descuento Flash,06/07/2023,2023-08-21,"7.261,51 €",Descuento
2,1003,CUPON,Black Friday,30/01/2023,2023-04-22,"3.542,64 €",Descuento
3,1004,CUPON,Rebajas Enero,23 Aug 2023,13/09/2023,"7.181,71 €",descuento
4,1005,Cupón,Cupon Verano,2023-04-24,22/09/2023,"3.834,60 €",descuento


**Hallazgos forenses en `campanas.csv`:**
* **1. Forma:** Dimensión de `45` campañas x `7` columnas, sin duplicados.
* **2. Tipos sucios:**
  * `coste` viene como texto con punto de miles, coma decimal y símbolo `€` (ej. `"5.397,33 €"`).
  * `fecha_inicio` y `fecha_fin` mezclan los 3 formatos temporales (incluyendo meses en inglés como `"23 Aug 2023"`).
* **3. Nulos:** `3` campañas sin `fecha_fin` (campañas continuas o aún activas).
* **4. Grafías y Regla de Cruce (Peligro de multiplicación):**
  * `canal` tiene **14 grafías** (`cupon`, `Cupón`, `CUPON`, `descuento`, `anuncios`, `Anuncios`, `publicidad`, `empresas`, `b2b`, `corporativo`) y **ninguna fila para SEO ni Referido** (que cuestan `0 €`).
  * `nombre` tiene espacios en blanco sobrantes al inicio/final (`"  Rebajas Enero "`).
  * **Advertencia para el CAC (Día 2):** Nunca cruzar `campanas` con `usuarios` directamente por `canal` sin agregar antes en CTEs independientes, o multiplicaremos cada campaña por miles de usuarios.


### 4. Auditoría: `suscripciones.csv` (Tabla de Hechos — Acto 3: Permanencia)
Registra los contratos de suscripción de cada alumno, qué plan eligió, su estado actual y si sigue activo o se dio de baja.


In [6]:
df_suscripciones = pd.read_csv(os.path.join(RAW_DIR, 'suscripciones.csv'), encoding='utf-8')
auditoria_tabla(df_suscripciones, 'suscripciones.csv', id_col='suscripcion_id')
df_suscripciones.head(5)


==================== AUDITORÍA: suscripciones.csv ====================
1. FORMA: 51,601 filas x 8 columnas
   -> Duplicados exactos de fila: 358
   -> Duplicados por clave primaria ('suscripcion_id'): 358 (IDs únicos: 51,243)

2. TIPOS DETECTADOS (Radiografía):
   - suscripcion_id         int64      (Ejemplo: np.int64(712710))
   - usuario_id             int64      (Ejemplo: np.int64(9963))
   - plan                   str        (Ejemplo: 'anual')
   - fecha_inicio           str        (Ejemplo: '2022-11-29')


   - fecha_fin              str        (Ejemplo: '9 Dec 2022')
   - precio                 str        (Ejemplo: '119,00 €')
   - estado                 str        (Ejemplo: 'Cancelada')
   - renovada               str        (Ejemplo: '0')

3. NULOS / VACÍOS (.isnull().sum()):
   - fecha_fin             : 17,417 nulos (33.75%)

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - plan                  : 8 valores distintos
     Frecuencias: {'mensual': 17582, 'Mensual': 9829, 'anual': 8389, 'MES': 4859, 'Anual': 4564, 'mes': 2805, 'año': 2309, '12m': 1264}
   - fecha_inicio          : 2276 valores distintos
   - fecha_fin             : 2282 valores distintos
   - precio                : 2 valores distintos
     Frecuencias: {'14,99 €': 35075, '119,00 €': 16526}


   - estado                : 8 valores distintos
     Frecuencias: {'cancelada': 15158, 'Cancelada': 9402, 'activa': 8701, 'baja': 6140, 'Activa': 5261, 'ok': 3455, 'Pausada': 1761, 'pausada': 1723}
   - renovada              : 4 valores distintos
     Frecuencias: {'0': 17722, 'No': 17553, 'si': 8195, '1': 8131}




,suscripcion_id,usuario_id,plan,fecha_inicio,fecha_fin,precio,estado,renovada
0,712710,9963,anual,2022-11-29,9 Dec 2022,"119,00 €",Cancelada,0
1,719631,15348,mensual,21/04/2023,10/07/2023,"14,99 €",Cancelada,No
2,737559,29342,mensual,26/05/2024,NaN,"14,99 €",activa,No
3,735768,27946,Anual,2023-05-27,NaN,"119,00 €",activa,1
4,740435,31597,mensual,03/05/2023,NaN,"14,99 €",Activa,1


**Hallazgos forenses en `suscripciones.csv`:**
* **1. Forma y Relación 1:N:** `51,601` filas x `8` columnas (`358` duplicados) para `40,000` usuarios únicos. Un mismo alumno puede tener más de una suscripción a lo largo del tiempo (~1,29 suscripciones por alumno).
* **2. El segundo nulo con significado (`fecha_fin`):**
  * Hay **17,417 suscripciones sin `fecha_fin` (33.75%)**.
  * Tal como advirtió Toni, **NO es un dato ausente ni roto**: son las **suscripciones activas** (alumnos que siguen pagando hoy, alrededor de 22.000 suscriptores activos). Si descartáramos las filas sin `fecha_fin`, borraríamos a los mejores alumnos de la empresa y hundiríamos el LTV de los canales que mejor retienen.
* **3. Tipos y Grafías:**
  * `precio` como texto con `€` y coma (`"14,99 €"`, `"119,00 €"`).
  * `plan` escrito de `8` formas (`mensual`, `Mensual`, `MES`, `mes`, `anual`, `Anual`, `año`, `12m`).
  * `estado` escrito de `8` formas (`activa`, `Activa`, `ACTIVA`, `cancelada`, `Cancelada`, `baja`, `Baja`, `Pausada`, `pausada`...).
  * `renovada` mezcla booleanos textuales y numéricos (`0`, `1`, `No`, `no`, `si`, `Sí`, `true`, `false`...).


### 5. Auditoría: `pagos.csv` (Tabla de Hechos — Acto 4: Valor y Renovación)
Exportación de Stripe con cada cobro realizado (`alta` o `renovacion`). Es donde vive el dinero real (el LTV) y el indicador de quién vuelve a pagar.


In [7]:
df_pagos = pd.read_csv(os.path.join(RAW_DIR, 'pagos.csv'), encoding='utf-8')
auditoria_tabla(df_pagos, 'pagos.csv', id_col='pago_id')
df_pagos.head(5)


==================== AUDITORÍA: pagos.csv ====================
1. FORMA: 171,459 filas x 6 columnas
   -> Duplicados exactos de fila: 1,360
   -> Duplicados por clave primaria ('pago_id'): 1,360 (IDs únicos: 170,099)

2. TIPOS DETECTADOS (Radiografía):
   - pago_id                int64      (Ejemplo: np.int64(9164136))
   - suscripcion_id         int64      (Ejemplo: np.int64(749432))
   - fecha                  str        (Ejemplo: '2023-06-06')
   - importe                str        (Ejemplo: '119,00 €')
   - tipo                   str        (Ejemplo: 'Alta')
   - metodo                 str        (Ejemplo: 'Bizum')

3. NULOS / VACÍOS (.isnull().sum()):
   - metodo                : 5,163 nulos (3.01%)

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - fecha                 : 2282 valores distintos
   - importe               : 17 valores distintos
     Frecuencias: {'14,99 €': 140603, '119,00 €': 15543, '4,99 €': 6205, '7,49 €': 3148, '35,70 €': 2431, '47,60 €': 1491, '9,99 €': 105

,pago_id,suscripcion_id,fecha,importe,tipo,metodo
0,9164136,749432,2023-06-06,"119,00 €",Alta,Bizum
1,9116430,735081,2022-11-19,"14,99 €",Renovación,Tarjeta
2,9092592,727950,21 Jan 2023,"14,99 €",Alta,NaN
3,9076834,723249,03/03/2023,"14,99 €",alta,tarjeta
4,9138523,741594,23/02/2024,"14,99 €",renovacion,Bizum


**Hallazgos forenses en `pagos.csv`:**
* **1. Forma y la Mina #1 de Toni (Duplicados de Stripe):**
  * `171,459` filas totales, pero solo **`170,099` `pago_id` únicos**.
  * Detectamos exactamente **`1,360` pagos duplicados** (`1,360` filas idénticas) provocados por el reintento del webhook de Stripe. Si sumáramos `importe` sin deduplicar por `pago_id`, inflaríamos artificialmente el LTV y los ingresos.
* **2. El Nulo Real (`metodo`):**
  * La columna `metodo` tiene **`5,163` vacíos (3.01%)**.
  * A diferencia de `campana_id` y `fecha_fin`, este **SÍ es un nulo real** (ausencia de registro del medio de pago). Se constata y se etiqueta como `'desconocido'`, sin eliminar la fila porque el cobro (`importe`) sí ocurrió y cuenta para el LTV.
* **3. Tipos y Grafías:**
  * `importe` en texto con `€` y coma (`"14,99 €"`, `"4,99 €"` de la primera cuota con cupón, `"119,00 €"`).
  * `tipo` tiene `8` grafías para distinguir el cobro inicial (`alta`, `Alta`, `ALTA`, `primera_cuota`...) de las renovaciones (`renovacion`, `Renovación`, `renov`, `recurrente`...).
  * `metodo` tiene `7` grafías (`Tarjeta`, `tarjeta`, `TARJETA`, `Bizum`, `bizum`, `PayPal`, `paypal`...).
* **4. Ruta de cruce:** `pagos` NO tiene `usuario_id` ni `canal`; se une a `suscripciones` mediante `suscripcion_id`, y de ahí a `usuarios` mediante `usuario_id`.


### 6. Auditoría: `matriculas.csv` (Tabla de Hechos — Acto 2: Aprendizaje)
Exportación del LMS con cada curso en el que se matricula un alumno, su porcentaje de avance, los minutos de vídeo consumidos y si lo completó.


In [8]:
df_matriculas = pd.read_csv(os.path.join(RAW_DIR, 'matriculas.csv'), encoding='utf-8')
auditoria_tabla(df_matriculas, 'matriculas.csv', id_col='matricula_id')
df_matriculas.head(5)


==================== AUDITORÍA: matriculas.csv ====================
1. FORMA: 249,635 filas x 8 columnas
   -> Duplicados exactos de fila: 1,981
   -> Duplicados por clave primaria ('matricula_id'): 1,981 (IDs únicos: 247,654)

2. TIPOS DETECTADOS (Radiografía):
   - matricula_id           int64      (Ejemplo: np.int64(3219949))
   - usuario_id             int64      (Ejemplo: np.int64(35519))
   - curso_id               int64      (Ejemplo: np.int64(5110))
   - fecha_matricula        str        (Ejemplo: '17 May 2024')
   - completado             str        (Ejemplo: 'false')
   - fecha_completado       str        (Ejemplo: '2024-06-30')
   - pct_completado         str        (Ejemplo: '55 %')
   - minutos_vistos         int64      (Ejemplo: np.int64(267))

3. NULOS / VACÍOS (.isnull().sum()):
   - fecha_completado      : 191,388 nulos (76.67%)

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - fecha_matricula       : 2286 valores distintos
   - completado            : 8 valores dis

,matricula_id,usuario_id,curso_id,fecha_matricula,completado,fecha_completado,pct_completado,minutos_vistos
0,3219949,35519,5110,17 May 2024,false,NaN,55 %,267
1,3027528,4402,5333,2024-03-29,no,NaN,28 %,39
2,3200387,32363,5278,2024-06-10,si,2024-06-30,81 %,194
3,3183945,29720,5215,2024-05-01,no,NaN,67 %,237
4,3169563,27371,5324,2023-12-23,1,2024-04-11,100 %,188


**Hallazgos forenses en `matriculas.csv`:**
* **1. Forma:** La tabla más grande del modelo con **`249,635` filas x `8` columnas** (`247,654` matrículas únicas, `1,981` duplicados). Escala coherente con las ~250.000 matrículas (~6,2 matrículas por alumno).
* **2. Tipos sucios:**
  * `pct_completado` viene como texto con espacio y símbolo `%` (`"55 %"`, `"100 %"`).
  * `completado` viene como texto mezclando múltiples representaciones booleanas (`si`, `Sí`, `true`, `1`, `no`, `No`, `false`, `0`...).
  * `fecha_matricula` y `fecha_completado` en texto con múltiples formatos.
* **3. Nulos esperables (`fecha_completado`):**
  * `191,388` filas (76.67%) no tienen `fecha_completado`: corresponde lógicamente a las matrículas que el alumno **no ha terminado** (`completado == False`).


### 7. Auditoría: `cursos.csv` (Dimensión de Contexto — Catálogo Educativo)
Ficha de los 600 cursos de la plataforma con su categoría, nivel y duración. Será clave el jueves para el control de confusión ante la hipótesis de Paula Rivas (¿es culpa de los cursos de programación o del canal?).


In [9]:
df_cursos = pd.read_csv(os.path.join(RAW_DIR, 'cursos.csv'), encoding='utf-8')
auditoria_tabla(df_cursos, 'cursos.csv', id_col='curso_id')
df_cursos.head(5)


==================== AUDITORÍA: cursos.csv ====================
1. FORMA: 600 filas x 7 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('curso_id'): 0 (IDs únicos: 600)

2. TIPOS DETECTADOS (Radiografía):
   - curso_id               int64      (Ejemplo: np.int64(5001))
   - titulo                 str        (Ejemplo: 'Python desde Cero 1')
   - categoria              str        (Ejemplo: 'programacion')
   - nivel                  str        (Ejemplo: 'avanzado')
   - n_lecciones            int64      (Ejemplo: np.int64(26))
   - duracion_horas         str        (Ejemplo: '10,0 h')
   - fecha_publicacion      str        (Ejemplo: '2021-02-16')

3. NULOS / VACÍOS (.isnull().sum()):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas de texto):
   - titulo                : 600 valores distintos
   - categoria             : 18 valores distintos
     Frecuencias: {'ofimatica': 38, 'idiomas': 38, 'business': 38, 'programacion':

,curso_id,titulo,categoria,nivel,n_lecciones,duracion_horas,fecha_publicacion
0,5001,Python desde Cero 1,programacion,avanzado,26,"10,0 h",2021-02-16
1,5002,JavaScript Moderno 1,programacion,basico,43,"22,0 h",2020-07-06
2,5003,SQL Práctico 1,dev,Intermedio,42,"13,7 h",2021-08-21
3,5004,React Avanzado 1,dev,Intermedio,33,"16,6 h",12 Apr 2020
4,5005,APIs con Node 1,dev,Intermedio,39,"13,6 h",22/10/2020


**Hallazgos forenses en `cursos.csv`:**
* **1. Forma:** `600` cursos x `7` columnas (`0` duplicados), cuadrando con el catálogo de 600 cursos de AulaViva.
* **2. Tipos y Grafías:**
  * `duracion_horas` viene como texto con coma decimal y sufijo `" h"` (ej. `"10,0 h"`).
  * `categoria` tiene **`18` grafías** para las 6 categorías oficiales (`programacion`, `Programación`, `dev`, `diseño`, `Diseño`, `marketing`, `ofimatica`, `idiomas`, `negocios`...).
  * `nivel` tiene **`8` grafías** (`principiante`, `Principiante`, `basico`, `intermedio`, `Intermedio`, `avanzado`, `Avanzado`...).
  * `titulo` presenta espacios sobrantes al inicio o final.


---
## Comprobación Forense de las 3 Minas de Toni Márquez vs. el Nulo Real
Antes de cerrar el Día 1, verificamos con código las tres advertencias de Toni para dimensionar con precisión el trabajo de limpieza de mañana:
1. **Mina 1 (Stripe):** Pagos duplicados por reintento del webhook.
2. **Mina 2 (Captación):** ¿Qué canales tienen `campana_id` vacío en `usuarios.csv`?
3. **Mina 3 (Permanencia):** ¿Qué estado tienen las suscripciones con `fecha_fin` vacía en `suscripciones.csv`?
4. **Contraste con Nulo Real:** Los vacíos en `pagos.metodo`.


In [10]:
print("1) MINA 1 — Duplicados en pagos.csv (Webhook de Stripe):")
print(f"   Filas totales en pagos: {len(df_pagos):,}")
print(f"   IDs únicos (pago_id):   {df_pagos['pago_id'].nunique():,}")
print(f"   Cobros duplicados:      {df_pagos['pago_id'].duplicated().sum():,}\n")

print("2) MINA 2 — Distribución de 'canal' (crudo) cuando campana_id es NULO en usuarios.csv:")
print(df_usuarios[df_usuarios['campana_id'].isnull()]['canal'].value_counts().to_dict())
print(f"   -> Total usuarios sin campana_id: {df_usuarios['campana_id'].isnull().sum():,}\n")

print("3) MINA 3 — Distribución de 'estado' (crudo) cuando fecha_fin es NULA en suscripciones.csv:")
print(df_suscripciones[df_suscripciones['fecha_fin'].isnull()]['estado'].value_counts().to_dict())
print(f"   -> Total suscripciones sin fecha_fin: {df_suscripciones['fecha_fin'].isnull().sum():,}\n")

print("4) NULO REAL — Distribución de 'metodo' en pagos.csv (incluyendo vacíos):")
print(df_pagos['metodo'].value_counts(dropna=False).to_dict())


1) MINA 1 — Duplicados en pagos.csv (Webhook de Stripe):
   Filas totales en pagos: 171,459
   IDs únicos (pago_id):   170,099
   Cobros duplicados:      1,360

2) MINA 2 — Distribución de 'canal' (crudo) cuando campana_id es NULO en usuarios.csv:
{'seo': 3373, 'referido': 2116, 'organico': 1863, 'SEO': 1472, 'buscador': 1292, 'Referido': 1064, 'amigo': 872, 'recomendacion': 748, 'cupon': 333, 'Cupón': 236, 'anuncios': 203, 'Anuncios': 131, 'empresas': 122, 'descuento': 115, 'CUPON': 84, 'ads': 72, 'b2b': 62, 'publicidad': 60, 'corporativo': 37}
   -> Total usuarios sin campana_id: 14,255

3) MINA 3 — Distribución de 'estado' (crudo) cuando fecha_fin es NULA en suscripciones.csv:
{'activa': 8701, 'Activa': 5261, 'ok': 3455}
   -> Total suscripciones sin fecha_fin: 17,417

4) NULO REAL — Distribución de 'metodo' en pagos.csv (incluyendo vacíos):
{'tarjeta': 34671, 'card': 34624, 'Tarjeta': 34351, 'PayPal': 20011, 'paypal': 19497, 'bizum': 11600, 'Bizum': 11542, nan: 5163}


---
## Síntesis del Día 1: Mapa del Modelo e Inventario de Suciedad (Hoja de Ruta Día 2)


In [11]:
inventario = [
    {
        'fichero': 'usuarios.csv',
        'rol_modelo': 'Hecho Central (Acto 1: Captación)',
        'filas_brutas': len(df_usuarios),
        'cols': df_usuarios.shape[1],
        'dups_pk': df_usuarios['usuario_id'].duplicated().sum(),
        'columnas_sucias': f"canal ({df_usuarios['canal'].nunique()} grafías), plan_inicial ({df_usuarios['plan_inicial'].nunique()}), ciudad ({df_usuarios['ciudad'].nunique()}), fecha_registro",
        'nulos_y_trampas': f"campana_id: {df_usuarios['campana_id'].isnull().sum():,} nulos CON SIGNIFICADO (SEO/Referido/sin campaña). No usar INNER JOIN."
    },
    {
        'fichero': 'canales.csv',
        'rol_modelo': 'Dimensión Principal (5 canales)',
        'filas_brutas': len(df_canales),
        'cols': df_canales.shape[1],
        'dups_pk': df_canales['canal_id'].duplicated().sum(),
        'columnas_sucias': f"tipo ({df_canales['tipo'].nunique()} grafías: Referido, B2B)",
        'nulos_y_trampas': 'Catálogo maestro de los 5 canales: cupon, anuncios, seo, referido, empresas.'
    },
    {
        'fichero': 'campanas.csv',
        'rol_modelo': 'Dimensión Coste (Acto 1: Captación)',
        'filas_brutas': len(df_campanas),
        'cols': df_campanas.shape[1],
        'dups_pk': df_campanas['campana_id'].duplicated().sum(),
        'columnas_sucias': f"coste (€ y coma), canal ({df_campanas['canal'].nunique()} grafías), fechas, espacios en nombre",
        'nulos_y_trampas': 'Solo canales de pago (sin SEO ni referido). Agregar por CTE antes de cruzar con usuarios.'
    },
    {
        'fichero': 'suscripciones.csv',
        'rol_modelo': 'Hecho Permanencia (Acto 3)',
        'filas_brutas': len(df_suscripciones),
        'cols': df_suscripciones.shape[1],
        'dups_pk': df_suscripciones['suscripcion_id'].duplicated().sum(),
        'columnas_sucias': f"precio (€), plan ({df_suscripciones['plan'].nunique()} grafías), estado ({df_suscripciones['estado'].nunique()}), renovada ({df_suscripciones['renovada'].nunique()}), fechas",
        'nulos_y_trampas': f"fecha_fin: {df_suscripciones['fecha_fin'].isnull().sum():,} nulos CON SIGNIFICADO = suscripción ACTIVA (sigue). No descartar."
    },
    {
        'fichero': 'pagos.csv',
        'rol_modelo': 'Hecho Valor / LTV (Acto 4)',
        'filas_brutas': len(df_pagos),
        'cols': df_pagos.shape[1],
        'dups_pk': df_pagos['pago_id'].duplicated().sum(),
        'columnas_sucias': f"importe (€), tipo ({df_pagos['tipo'].nunique()} grafías), metodo ({df_pagos['metodo'].nunique()} grafías), fecha",
        'nulos_y_trampas': f"{df_pagos['pago_id'].duplicated().sum():,} pagos duplicados (webhook Stripe). metodo: {df_pagos['metodo'].isnull().sum():,} nulos REALES ('desconocido')."
    },
    {
        'fichero': 'matriculas.csv',
        'rol_modelo': 'Hecho Aprendizaje (Acto 2)',
        'filas_brutas': len(df_matriculas),
        'cols': df_matriculas.shape[1],
        'dups_pk': df_matriculas['matricula_id'].duplicated().sum(),
        'columnas_sucias': f"pct_completado (%), completado ({df_matriculas['completado'].nunique()} grafías booleanas), fechas",
        'nulos_y_trampas': f"fecha_completado: {df_matriculas['fecha_completado'].isnull().sum():,} vacíos (cursos no terminados)."
    },
    {
        'fichero': 'cursos.csv',
        'rol_modelo': 'Dimensión Contexto (600 cursos)',
        'filas_brutas': len(df_cursos),
        'cols': df_cursos.shape[1],
        'dups_pk': df_cursos['curso_id'].duplicated().sum(),
        'columnas_sucias': f"duracion_horas ('h' y coma), categoria ({df_cursos['categoria'].nunique()} grafías), nivel ({df_cursos['nivel'].nunique()}), fecha_publicacion",
        'nulos_y_trampas': 'Clave para el control de confusión por categoría del jueves (hipótesis de Paula).'
    }
]

df_inventario = pd.DataFrame(inventario)
df_inventario


,fichero,rol_modelo,filas_brutas,cols,dups_pk,columnas_sucias,nulos_y_trampas
0,usuarios.csv,Hecho Central (Acto 1: Captación),40000,6,0,"canal (19 grafías), plan_inicial (8), ciudad (...","campana_id: 14,255 nulos CON SIGNIFICADO (SEO/..."
1,canales.csv,Dimensión Principal (5 canales),5,3,0,"tipo (4 grafías: Referido, B2B)","Catálogo maestro de los 5 canales: cupon, anun..."
2,campanas.csv,Dimensión Coste (Acto 1: Captación),45,7,0,"coste (€ y coma), canal (14 grafías), fechas, ...",Solo canales de pago (sin SEO ni referido). Ag...
3,suscripciones.csv,Hecho Permanencia (Acto 3),51601,8,358,"precio (€), plan (8 grafías), estado (8), reno...","fecha_fin: 17,417 nulos CON SIGNIFICADO = susc..."
4,pagos.csv,Hecho Valor / LTV (Acto 4),171459,6,1360,"importe (€), tipo (8 grafías), metodo (7 grafí...","1,360 pagos duplicados (webhook Stripe). metod..."
5,matriculas.csv,Hecho Aprendizaje (Acto 2),249635,8,1981,"pct_completado (%), completado (8 grafías bool...","fecha_completado: 191,388 vacíos (cursos no te..."
6,cursos.csv,Dimensión Contexto (600 cursos),600,7,0,"duracion_horas ('h' y coma), categoria (18 gra...",Clave para el control de confusión por categor...


### Conclusión al cerrar el Día 1
* **El mapa está completo:** 40.000 alumnos en el centro (`usuarios.csv`), 5 canales (`canales.csv`), 45 campañas de pago (`campanas.csv`), ~51.600 suscripciones (`suscripciones.csv`), ~171.460 registros de pago (`pagos.csv`) y ~249.600 matrículas (`matriculas.csv`) sobre 600 cursos (`cursos.csv`).
* **Las 3 minas están localizadas y cuantificadas:**
  1. **Duplicados de Stripe** en `pagos.csv` listos para deduplicar por `pago_id`.
  2. **`campana_id` vacío** confirmado como alta sin campaña de pago (SEO/Referido), lo que exige `LEFT JOIN`.
  3. **`fecha_fin` vacía** confirmada al 100% como suscripción `activa` (alumnos que siguen en la plataforma).
* **Siguiente paso (Día 2):** Limpieza integral de las 7 tablas en Pandas, exportación a `data/clean/`, definición formal de la métrica de **Valor por canal** y construcción en SQL del ranking de superficie que mira Growth.
